# Leakage-safe baseline models

This notebook establishes the first reproducible model comparison (EXP-001). It creates historical training cases only where every return in the 30-minute outcome window is observed, then evaluates a constant prior and two simple classifiers chronologically. It reads the supplied inputs but never modifies them.

## Evaluation design

For a candidate target row `t`, the label is whether the cumulative return from rows `t-29` through `t` is positive: this is the 30-minute price move from minute 0 to minute 30. The feature window is rows `t-29` through `t-15`—the first 15 one-minute returns—so it ends 15 minutes before the target.

Candidates are taken every five hours when all 30 returns are present, roughly matching the competition case spacing while avoiding heavily overlapping labels. They are split in time into 70% train, 15% validation, and 15% final test. A 30-minute embargo is enforced at each boundary. The test interval is not used to tune models.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
PROBABILITY_EPSILON = 1e-6
CASE_SPACING_MINUTES = 5 * 60
EMBARGO_MINUTES = 30

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "epfl-vol-hack/dataset.csv"
assert DATA_PATH.is_file(), f"Missing input: {DATA_PATH.resolve()}"

In [ ]:
returns = pd.read_csv(DATA_PATH)
assert returns.columns.tolist() == ["timestamp", "ret"]
assert returns["timestamp"].is_monotonic_increasing
assert np.all(np.diff(returns["timestamp"].to_numpy()) == 60)

ret = returns["ret"].to_numpy()
observed = np.isfinite(ret)

# The complete label window is t-29 ... t (30 one-minute returns).
candidate_rows = np.arange(30, len(returns), CASE_SPACING_MINUTES)
observed_prefix = np.r_[0, np.cumsum(observed)]
complete_window = (
    observed_prefix[candidate_rows + 1] - observed_prefix[candidate_rows - 29] == 30
)
target_rows = candidate_rows[complete_window]

return_prefix = np.r_[0.0, np.cumsum(np.nan_to_num(ret, nan=0.0))]
thirty_minute_move = (
    return_prefix[target_rows + 1] - return_prefix[target_rows - 29]
)
non_tie = thirty_minute_move != 0
target_rows = target_rows[non_tie]
thirty_minute_move = thirty_minute_move[non_tie]
labels = (thirty_minute_move > 0).astype(int)
target_times = pd.to_datetime(
    returns.loc[target_rows, "timestamp"].to_numpy(), unit="s", utc=True
)

pd.Series(
    {
        "eligible historical cases": len(target_rows),
        "first target (UTC)": target_times.min(),
        "last target (UTC)": target_times.max(),
        "positive-label share": labels.mean(),
    },
    name="value",
).to_frame()

## Features available at prediction time

The raw 15 observed returns are retained, with simple cumulative-return, dispersion, sign, and short-window momentum summaries. No return at or after `t-14` is used in the feature matrix.

In [ ]:
# Columns run oldest to newest: t-29, ..., t-15.
visible_offsets = np.arange(29, 14, -1)
visible_rows = target_rows[:, None] - visible_offsets
assert np.all(visible_rows <= target_rows[:, None] - 15)
visible_returns = ret[visible_rows]
assert np.isfinite(visible_returns).all()

sign_changes = (np.diff(np.signbit(visible_returns), axis=1) != 0).sum(axis=1)
feature_matrix = np.column_stack(
    [
        visible_returns,
        visible_returns.sum(axis=1),
        visible_returns.mean(axis=1),
        visible_returns.std(axis=1),
        np.abs(visible_returns).sum(axis=1),
        (visible_returns > 0).mean(axis=1),
        visible_returns[:, -3:].sum(axis=1),
        visible_returns[:, -5:].sum(axis=1),
        visible_returns[:, -10:].sum(axis=1),
        visible_returns.min(axis=1),
        visible_returns.max(axis=1),
        sign_changes,
    ]
)

feature_names = [f"ret_t_minus_{offset}" for offset in visible_offsets]
feature_names += [
    "return_sum_15",
    "return_mean_15",
    "return_std_15",
    "absolute_return_sum_15",
    "positive_return_share_15",
    "return_sum_last_3",
    "return_sum_last_5",
    "return_sum_last_10",
    "return_min_15",
    "return_max_15",
    "sign_changes_15",
]

pd.Series(feature_names, name="feature").to_frame()

In [ ]:
n_cases = len(target_rows)
validation_start = target_times[int(n_cases * 0.70)]
test_start = target_times[int(n_cases * 0.85)]
embargo = pd.Timedelta(minutes=EMBARGO_MINUTES)

train_mask = target_times < validation_start - embargo
validation_mask = (target_times >= validation_start) & (
    target_times < test_start - embargo
)
test_mask = target_times >= test_start
assert not (train_mask & validation_mask).any()
assert not (validation_mask & test_mask).any()
assert target_times[train_mask].max() < target_times[validation_mask].min() - embargo
assert target_times[validation_mask].max() < target_times[test_mask].min() - embargo

split_summary = pd.DataFrame(
    [
        ("train", train_mask),
        ("validation", validation_mask),
        ("test", test_mask),
    ],
    columns=["split", "mask"],
).assign(
    cases=lambda frame: frame["mask"].map(np.sum),
    first_target=lambda frame: frame["mask"].map(lambda mask: target_times[mask].min()),
    last_target=lambda frame: frame["mask"].map(lambda mask: target_times[mask].max()),
    positive_share=lambda frame: frame["mask"].map(lambda mask: labels[mask].mean()),
).drop(columns="mask")
split_summary

## Baselines and metrics

All models are fitted on the training period only. The constant prior is the training positive rate. Logistic regression uses standardised features; histogram gradient boosting is deliberately shallow and has no internal random validation split. Probabilities are clipped strictly inside `(0, 1)` before log-loss scoring. Lower log loss and Brier score are better; AUC measures ranking only.

In [ ]:
models = {
    "Constant prior": None,
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.1, max_iter=1_000, random_state=RANDOM_SEED),
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.05,
        max_leaf_nodes=7,
        min_samples_leaf=50,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=RANDOM_SEED,
    ),
}
split_masks = {"validation": validation_mask, "test": test_mask}
training_prior = labels[train_mask].mean()
results = []
predictions = {}

for model_name, model in models.items():
    if model is not None:
        model.fit(feature_matrix[train_mask], labels[train_mask])

    for split_name, split_mask in split_masks.items():
        if model is None:
            probability = np.full(split_mask.sum(), training_prior)
        else:
            probability = model.predict_proba(feature_matrix[split_mask])[:, 1]

        probability = np.clip(probability, PROBABILITY_EPSILON, 1 - PROBABILITY_EPSILON)
        predictions[(model_name, split_name)] = probability
        results.append(
            {
                "model": model_name,
                "split": split_name,
                "log_loss": log_loss(labels[split_mask], probability),
                "brier_score": brier_score_loss(labels[split_mask], probability),
                "roc_auc": roc_auc_score(labels[split_mask], probability),
                "mean_probability": probability.mean(),
            }
        )

results = pd.DataFrame(results).sort_values(["split", "log_loss"])
results.round(6)

In [ ]:
comparison = results.pivot(index="model", columns="split", values="log_loss")
comparison["test_improvement_vs_prior"] = (
    comparison.loc["Constant prior", "test"] - comparison["test"]
)
comparison.sort_values("test")

## What this establishes

- The held-out test score, rather than the validation score, is the first estimate of generalisation.
- If the boosted model improves test log loss over logistic regression, the gain is evidence that simple non-linear interactions may help; it is not yet a license to tune repeatedly on the test period.
- The next experiment should choose calibration and additional long-history features using the validation period, then make one final untouched-test comparison.
- Any production submission must rebuild features at each supplied ID using only rows through `ID - 15 minutes`, with finite, clipped probabilities.